In [ ]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
from src.models.codegen_loader_small_model import CodeGenModelWrapper

In [ ]:
def train_codegen_lora(
    model_wrapper: CodeGenMultiModelWrapper,
    train_samples: list,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 2
):
    """Fine-tuning Salesforce/codegen-350M-multi via PyTorch and LoRA"""
    def tokenize_fn(examples):
        return model_wrapper.tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

    raw_dataset = Dataset.from_dict({"text": train_samples})
    tokenized_dataset = raw_dataset.map(tokenize_fn, batched=True)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        logging_steps=5,
        save_strategy="epoch",
        learning_rate=3e-4,
        fp16=torch.cuda.is_available(),
        use_cpu=not torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model_wrapper.model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=model_wrapper.tokenizer, mlm=False)
    )

    trainer.train()
    model_wrapper.save_lora_weights(output_dir)